# Глава 14: полные версии листингов из `labs/ch14`

**Вопрос.** В главе 14 длинные листинги сокращены: в тексте остались ручной расчёт и короткая функция,
а целиком код лежит в каталоге `labs/ch14`. Проверяют ли полные версии то же, что обещает текст?

**Прогноз.** Запишите до запуска:

1. сколько токенов оставят top-k с $k=3$, ядерная выборка с $p=0{,}9$ и порог от лидера $0{,}2$ на строке логитов стенда;
2. сколько байт займёт блок MXFP4 из 32 весов и сколько это бит на вес;
3. во что превратится выход внимания, если в блочном softmax забыть поправку $\alpha$.

Ноутбук проходит восемь файлов в том порядке, в котором механизмы встречаются в главе. Каждый файл
загружается по явному пути через `importlib`, поэтому у него своё пространство имён и никакого скрытого
состояния между ячейками нет. Всё работает на CPU за несколько секунд, сервер и GPU не нужны.

In [ ]:
import hashlib
import importlib.util
import platform
import random
import sys
from pathlib import Path

import numpy as np
import torch

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

print("Python", platform.python_version(), "| torch", torch.__version__, "| numpy", np.__version__)


def find_labs_ch14():
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "labs" / "ch14"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError("labs/ch14 не найден: запустите ноутбук внутри репозитория книги")


LABS_CH14 = find_labs_ch14()


def load_lab(name):
    # Загружаем файл как отдельный модуль. Регистрация в sys.modules нужна dataclass
    # из stream_metrics: без неё Python не найдёт модуль, в котором объявлен класс.
    lab_path = LABS_CH14 / f"{name}.py"
    module_name = f"ch14_{name}"
    spec = importlib.util.spec_from_file_location(module_name, lab_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)   # здесь выполняются проверки уровня модуля
    return module


print("labs:", LABS_CH14, "| файлов:", len(list(LABS_CH14.glob("*.py"))))

## 1. Фильтры хвоста: top-k, ядерная выборка, порог от лидера

Раздел главы «От логитов к токену», подраздел «Отсечение хвоста» (`sec:inf-topk-topp-minp`), рисунок
`fig:sampling-filters`. В книге одна функция на все три правила. В файле
`labs/ch14/sampling_filters.py` у каждого правила своя маска, общая перенормировка и тысяча случайных
выборок. Проверки срабатывают при загрузке файла: правила оставляют 3, 5 и 4 токена, а выборка после
ядерного фильтра ни разу не попадает в отрезанный хвост.

In [ ]:
torch.manual_seed(0)
sampling = load_lab("sampling_filters")

kept = {name: int(mask.sum()) for name, mask in sampling.masks.items()}
print("оставлено токенов:", kept)
assert kept == {"top_k": 3, "top_p": 5, "min_p": 4}

## 2. KV-кэш настоящей BookLM

Раздел «Кэш ключей и значений» (`sec:inf-kv-cache`), опыт «KV-кэш обученной BookLM».
В книге кэш BookLM проверен на одном тексте. В `labs/ch14/kv_cache_attention.py` та же проверка идёт на классе
`BookLM` из `booklm/model.py` со случайными весами, с пакетом $B=2$ и кусками по 1, 3, 5 и 23 токена, а в конце
нарочно забывает сдвинуть `cache["len"]`: формы те же, логиты расходятся.

In [ ]:
kv = load_lab("kv_cache_attention")
kv.main()

## 3. Блок MXFP4: коды E2M1, масштаб E8M0 и 17 байт

Раздел «Квантование», подраздел «MXFP4: четыре бита с плавающей точкой и общий масштаб» (`sec:inf-mxfp4`).
В книге оставлены кодирование кода и блока. В `labs/ch14/mxfp4_block.py` добавлены декодер, упаковка
в байты и порча: показатель масштаба выбирается через `round` вместо `floor`. Блок из 32 весов
занимает 16 байт кодов и байт масштаба, то есть $17 \cdot 8 / 32 = 4{,}25$ бита на вес.

In [ ]:
mxfp4 = load_lab("mxfp4_block")

print("байт масштаба:", hex(mxfp4.scale_byte), "| первые коды:", [hex(c) for c in mxfp4.codes[:4]])
print("размер блока:", len(mxfp4.packed), "байт =", len(mxfp4.packed) * 8 / mxfp4.BLOCK, "бита на вес")
print("ошибка floor:", round(mxfp4.sse(mxfp4.block, mxfp4.restored), 4),
      "| ошибка round:", round(mxfp4.sse(mxfp4.block, mxfp4.bad), 4))
assert len(mxfp4.packed) == 17

## 4. GGUF: разборщик и писатель на стандартной библиотеке

Раздел «GGUF: разбираем файл, а не расширение» (`sec:inf-gguf`). В книге короткий разбор одного тензора
без свойств. В `labs/ch14/gguf_reader.py` разборщик читает свойства всех типов (строки, числа, массивы),
проверяет выравнивание, а писатель собирает минимальный файл GGUF v3 с матрицей $2\times2$. Файл
создаётся во временном каталоге и после проверки исчезает. Последний опыт портит сигнатуру и ждёт
понятную ошибку вместо мусора.

In [ ]:
gguf = load_lab("gguf_reader")

print("данные тензора из файла:", gguf.values)
assert gguf.values == (1.0, 2.0, 3.0, 4.0)
assert gguf.align_up(33, 32) == 64

## 5. Блочный softmax и внимание без полной матрицы вероятностей

Раздел «Как не хранить матрицу внимания целиком: FlashAttention» (`sec:inf-flashattention`), листинги
«Потоковый softmax по блокам» и «Точное причинное внимание с потоковым softmax». Файл
`labs/ch14/flashattention_tiled.py` повторяет ручной расчёт на четырёх оценках
(0,702 с поправкой и 0,856 без неё) и сравнивает блочное внимание с наивным: неполный последний блок,
сдвиг всех логитов, входы в половинной точности. Проверки запускает функция `main()`.

In [ ]:
flash = load_lab("flashattention_tiled")
flash.main()

## 6. Распределитель страниц KV-кэша

Раздел «PagedAttention и vLLM» (`sec:inf-paged-attention`), подраздел «Учебный распределитель страниц».
Файл `labs/ch14/kv_block_pool.py` повторяет пример с рисунка (запрос A получает страницы 4, 1, 3) и
сравнивает страницы по 16 токенов с резервом на максимальную длину. Страницу кэша vLLM называет блоком,
отсюда имя класса `BlockPool`. Непрерывный планировщик поверх него — в `labs/ch14/continuous_scheduler.py`.

In [ ]:
pool_lab = load_lab("kv_block_pool")
pool_lab.main()

pool = pool_lab.BlockPool(n_blocks=3, block_size=4)
pool.add_request("X")
for _ in range(12):
    pool.append_token("X")
assert pool.address("X", 11) == (pool.tables["X"][2], 3)

## 7. Измерение запроса: TTFT, TPOT, ITL и перцентили

Раздел «Измерение скорости ответа» (`sec:inf-measurement-lab`). Файл
`labs/ch14/stream_metrics.py` содержит клиент потока, расчёт метрик, замкнутую и разомкнутую нагрузку
и сводку по перцентилям. Без сервера `main()` проверяет всё на синтетических журналах. С запущенным
OpenAI-совместимым сервером (например, `vllm serve`) вызовите `metrics.run_closed_loop()` или
`metrics.run_open_loop()` вручную: этот опыт в автоматическую проверку не входит.

In [ ]:
metrics = load_lab("stream_metrics")
metrics.main()

ms = 1_000_000
trace = metrics.StreamTrace(
    started_ns=0, first_content_ns=80 * ms, finished_ns=400 * ms,
    content_events_ns=[80 * ms, 180 * ms, 380 * ms], text="",
)
row = metrics.summarize(trace)
print(row)
assert row["ttft_ms"] == 80 and row["itl_chunk_ms"] == [100, 200]

## 8. Отложенный граф в духе ggml

Раздел «Как устроена llama.cpp» (`sec:inf-llamacpp`), подраздел «Граф вместо последовательности
C++-вызовов». Файл `labs/ch14/lazy_graph_calls.py` строит граф без вычислений, считает его функцией
`evaluate`, проверяет, что общий вход считается один раз, и оценивает, сколько промежуточных буферов
живут одновременно.

In [ ]:
graph = load_lab("lazy_graph_calls")

print("журнал операций:", graph.calls)
print("пик живых буферов в цепочке из трёх операций:", graph.peak_live_buffers(graph.c))
assert graph.calls.count("matmul") == 1

## Сломанная вариация

Без поправки $\alpha$ блочный softmax складывает доли, посчитанные от разных максимумов. Формы
тензоров остаются правильными, поэтому ошибку не поймает ни одна проверка формы. Её видно только при
сравнении с эталоном.

In [ ]:
torch.manual_seed(7)
q, k, v = (torch.randn(129, 64) for _ in range(3))
reference = flash.naive_causal_attention(q, k, v)
broken = flash.tiled_causal_attention(q, k, v, block_size=32, use_alpha=False)

gap = (broken - reference).abs().max().item()
print(f"формы совпадают: {broken.shape == reference.shape}, наибольшее расхождение: {gap:.3f}")
assert broken.shape == reference.shape and gap > 1e-2

**Задание.** Замените в `BlockPool.append_token` условие `length % self.block_size == 0` на
`length % self.block_size == 1`. Предскажите, на какой позиции метод `address` упадёт, и проверьте.

## Манифест

Контрольные суммы фиксируют, какие версии файлов прошли проверку в этом запуске.

In [ ]:
manifest = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()[:16]
    for path in sorted(LABS_CH14.glob("*.py"))
}
for name, digest in manifest.items():
    print(f"{digest}  {name}")
USED = ["flashattention_tiled", "gguf_reader", "kv_block_pool", "kv_cache_attention",
        "lazy_graph_calls", "mxfp4_block", "sampling_filters", "stream_metrics"]   # файлы, загруженные выше
assert all(f"{name}.py" in manifest for name in USED)

## Что соответствует чему в промышленных системах

| Учебный объект | Промышленный аналог |
|---|---|
| `sampling_filters.top_k_mask`, `top_p_mask`, `min_p_mask` | параметры `top_k`, `top_p`, `min_p` в `SamplingParams` vLLM и цепочка сэмплеров llama.cpp |
| `CachedSelfAttention` и `cache` | `past_key_values` в Transformers |
| `mxfp4_encode_block`, `pack_block` | веса MXFP4 в открытых моделях gpt-oss |
| `inspect_gguf`, `write_tiny_gguf` | `gguf_init_from_file` в ggml и пакет `gguf` в Python |
| `tiled_causal_attention` | ядра FlashAttention и `scaled_dot_product_attention` в PyTorch |
| `BlockPool`, `scheduler_tick` | менеджер блоков KV-кэша и планировщик vLLM |
| `summarize`, `run_open_loop` | `vllm bench serve` и другие генераторы нагрузки |
| `Node`, `evaluate`, `peak_live_buffers` | `ggml_cgraph`, `ggml_graph_compute` и распределитель `ggml_gallocr` |